In [2]:
import csv

lectura = open("CYP2D6_Diplotype_Phenotype_Table.xlsx")

In [ ]:
from Bio import Entrez

Entrez.email=""
Entrez.api_key=""

# Polimorfismos de estudio

variants = {"DPYD*2A": "rs3918290", "DPYD*13": "rs55886062", "DPYD_D949V": "rs67376798", "DPYD_HapB3": "rs56038477", "CYP2D6*3": "rs35742686", 
            "CYP2D6*4": "rs3892097", "CYP2D6*17": "rs28371706", "CYP2D6*41": "rs28371725", "UGT1A1*80": "rs887826"}

# Modificación para hacerlo para todas las variantes
import pandas as pd

# Diccionario que tomaremos como final
base_og = []
variantes = []

for k, v in variants.items():
    # Tomamos los datos del polimorfismo de la base de datos dbSNP
    fetch_handle = Entrez.esummary(db = "snp", id = v[2:], retmode = "xml")
    datos = Entrez.read(fetch_handle)
    fetch_handle.close()

    # Extraemos las distintas mutaciones que pueden causar esta variante
    resumen = datos["DocumentSummarySet"]["DocumentSummary"][0]
    mutaciones = resumen["SPDI"].split(",")
    bases = list(map(lambda x: (x[-2], "-") if resumen["SNP_CLASS"] == "del" else (x[-3], x[-1]) , mutaciones))

    # Bases original y mutadas
    base_og.append(bases[0][0])

    gen_mut = []
    for b in bases:
        gen_mut.append(b[1]) 

    variantes.append(gen_mut)
    
df_resumen = pd.DataFrame({"Polimorfismo": variants.keys(), "Base ancestral": base_og, "Mutaciones": variantes})
display(df_resumen)

In [ ]:
ancestral = dict(zip(df_resumen["Polimorfismo"], df_resumen["Base ancestral"])) #Lo onvierte en un dict tipo: {'CYP2D6*17': 'G', 'DPYD*2A': 'C', ...} para comparar base ancestral


#Sacar nombre del alelo mutado
def etiqueta_mutada(polimorfismo):
    """ Convierte p.ej "CYP2D6*17" a "*17" """
    if "*" in polimorfismo:
        return "*" + polimorfismo.split("*")[1]
    return "*" +polimorfismo.split("_")[1] #HapB3 p.ej tiene _ en vez de *


#Sacar el nombre del gen
def gen_sin_alelo(polimorfismo):
    """ Convierte p.ej "CYP2D6*17" a "CYP2D6" """
    if "*" in polimorfismo:
        return polimorfismo.split("*")[0]
    return polimorfismo.split("_")[0] 


#Función para convertir genotipo a números
def letras_a_nums(polimorfismo, genotipo):
    """ Convierte p.ej "G/A" a "*1/*17" """
    wild_type = ancestral[polimorfismo]
    mut = etiqueta_mutada(polimorfismo)
    alelos = ["*1" if b == wild_type else mut for b in genotipo.split("/")]
    alelos.sort(key = lambda a:a != "*1") #Para que el alelo siempre sea con la estructura *1/*17
    return alelos[0] + "/" + alelos[1]

In [ ]:
with open("Genotype Matrix.csv", mode = "r") as file:
    lector = csv.reader(file, delimiter=";")
    pacientes = {}
    cabecera = next(lector) #Para saltar el encabezado
    for fila in lector: 
        cod_paciente = fila[0]
        pacientes[cod_paciente] = {} #Por cada paciente se crea un diccionario
        for columna, elemento in zip(cabecera, fila):
            if columna != "Sample/Assay": #Para no guardar el ID del paciente ya que ya es la clave
                pacientes[cod_paciente][columna] = elemento

    print(pacientes)

#Para juntar las mutaciones d eun mismo gen, yo añadiría un, si empieza por CYP2D6 añadir a lista CYP2D6 y luego que sea la lista lo que añadimos en el diccionario
            


            
    

{'DPD932': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/C', 'CYP2D6*17': 'G/A', 'CYP2D6*41': 'C/C', 'DPYD*2A': 'C/C', 'DPYD*13': 'A/A', 'DPYD_HapB3': 'C/T', 'DPYD_D949V': 'T/T', 'UGT1A1*80': 'C/C'}, 'DPD933': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/C', 'CYP2D6*17': 'A/A', 'CYP2D6*41': 'C/C', 'DPYD*2A': 'C/C', 'DPYD*13': 'A/A', 'DPYD_HapB3': 'C/C', 'DPYD_D949V': 'T/T', 'UGT1A1*80': 'C/T'}, 'DPD934': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/T', 'CYP2D6*17': 'G/G', 'CYP2D6*41': 'C/C', 'DPYD*2A': 'C/C', 'DPYD*13': 'A/A', 'DPYD_HapB3': 'C/C', 'DPYD_D949V': 'T/T', 'UGT1A1*80': 'C/C'}, 'DPD935': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/C', 'CYP2D6*17': 'G/G', 'CYP2D6*41': 'C/C', 'DPYD*2A': 'C/C', 'DPYD*13': 'A/A', 'DPYD_HapB3': 'C/C', 'DPYD_D949V': 'T/T', 'UGT1A1*80': 'C/T'}, 'DPD936': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/C', 'CYP2D6*17': 'G/G', 'CYP2D6*41': 'C/T', 'DPYD*2A': 'C/C', 'DPYD*13': 'A/A', 'DPYD_HapB3': 'C/C', 'DPYD_D949V': 'T/T', 'UGT1A1*80': 'C/T'}, 'DPD937': {'CYP2D6*3': 'T/T', 'CYP2D6*4': 'C/C', 'CYP2D6*17

In [ ]:
#Obtener diccionario de números
pacientes_alelos_nums = {}

for paciente, genotipos in pacientes.items():
    pacientes_alelos_nums[paciente] = {}

    for poli, geno in genotipos.items():
        gen = gen_sin_alelo(poli)
        alelos = letras_a_nums(poli, geno)

        if gen not in pacientes_alelos_nums[paciente]:
            pacientes_alelos_nums[paciente][gen] = []
        pacientes_alelos_nums[paciente][gen].append(alelos)

In [ ]:
#Data Frame para visualizarlo bien
pacientes_conv = {}
for paciente,genotipos in pacientes.items():
    alelos_paciente = {}

    for polimorfismo, genotipo in genotipos.items():
        alelos_paciente[polimorfismo] = letras_a_nums(polimorfismo, genotipo)

    pacientes_conv[paciente] = alelos_paciente

df_alelos = pd.DataFrame.from_dict(pacientes_conv, orient = "index")
display(df_alelos)